In [1]:
from typing import List, Tuple
import numpy as np
from dataclasses import dataclass

@dataclass
class TwoLevelUnitary:
    """A 2-level unitary acting on a d-dimensional space."""
    dim: int
    i: int
    j: int
    submatrix: np.ndarray
    
    def to_full_matrix(self) -> np.ndarray:
        """Embed the 2x2 submatrix into the full d×d space."""
        U = np.eye(self.dim, dtype=complex)
        U[self.i, self.i] = self.submatrix[0, 0]
        U[self.i, self.j] = self.submatrix[0, 1]
        U[self.j, self.i] = self.submatrix[1, 0]
        U[self.j, self.j] = self.submatrix[1, 1]
        return U
    
    def dagger(self) -> 'TwoLevelUnitary':
        """Return the adjoint (conjugate transpose)."""
        return TwoLevelUnitary(
            dim=self.dim,
            i=self.i,
            j=self.j,
            submatrix=self.submatrix.conj().T
        )


def V_su2(a: complex, b: complex) -> np.ndarray:
    """
    Construct V(a,b) ∈ SU(2) such that V(a,b) @ [a, b]^T = [sqrt(|a|²+|b|²), 0]^T.
    
    From Eq. (D1):
        V(a,b) = (|a|² + |b|²)^{-1/2} * [[a*, b*], [-b, a]]
    """
    norm_sq = np.abs(a)**2 + np.abs(b)**2
    
    if norm_sq < 1e-14:
        return np.eye(2, dtype=complex)
    
    norm = np.sqrt(norm_sq)
    V = np.array([
        [np.conj(a), np.conj(b)],
        [-b, a]
    ], dtype=complex) / norm
    
    return V


def sud_decompose(U: np.ndarray, tol: float = 1e-12) -> List[TwoLevelUnitary]:
    """
    Decompose a d×d SU(d) matrix into a product of 2-level SU(2) unitaries.
    
    Algorithm from Appendix D of arXiv:2309.11051.
    
    Returns:
        List of TwoLevelUnitary objects [U_1, U_2, ..., U_k] such that
        U = U_1 @ U_2 @ ... @ U_k
    """
    d = U.shape[0]
    
    # Validate input
    if U.shape != (d, d):
        raise ValueError("Input must be a square matrix")
    if not np.allclose(U @ U.conj().T, np.eye(d), atol=tol):
        raise ValueError("Input matrix is not unitary")
    if not np.isclose(np.linalg.det(U), 1.0, atol=tol):
        raise ValueError(f"Input matrix is not special unitary (det = {np.linalg.det(U)})")
    
    # return empty list for identity
    if np.allclose(U, np.eye(d), atol=tol):
        return []
    # Base case: d = 2
    if d == 2:
        return [TwoLevelUnitary(dim=2, i=0, j=1, submatrix=U.copy())]
    
    # Recursive case: d > 2
    W = U.copy().astype(complex)
    v_gates: List[TwoLevelUnitary] = []
    
    # We want to transform first column [a_1, a_2, ..., a_d]^T to [1, 0, ..., 0]^T
    # 
    # Step 1: Apply V(a_1, a_2)_{0,1} to get [sqrt(|a_1|^2+|a_2|^2), 0, a_3, ..., a_d]^T
    # Step 2: Apply V(W[0,0], a_3)_{0,2} to zero out position 2
    # ... and so on
    #
    # After each V application, W[0,0] contains the accumulated norm
    
    for k in range(1, d):
        # Current state: W[0,0] has accumulated value, W[k,0] is to be zeroed
        a = W[0, 0]  # The accumulated value at position 0
        b = W[k, 0]  # The value to zero out
        
        # Build V(a, b) acting on subspace {0, k}
        V_2x2 = V_su2(a, b)
        
        # Create 2-level unitary
        V_gate = TwoLevelUnitary(dim=d, i=0, j=k, submatrix=V_2x2)
        v_gates.append(V_gate)
        
        # Apply V to W (left multiply): affects rows 0 and k
        # More efficient than full matrix multiplication
        row_0 = W[0, :].copy()
        row_k = W[k, :].copy()
        
        W[0, :] = V_2x2[0, 0] * row_0 + V_2x2[0, 1] * row_k
        W[k, :] = V_2x2[1, 0] * row_0 + V_2x2[1, 1] * row_k
    
    # Now W should have the form [[1, 0...], [0, W']] where W' ∈ SU(d-1)
    # Verify
    if not np.isclose(np.abs(W[0, 0]), 1.0, atol=tol):
        raise RuntimeError(f"Failed to reduce: |W[0,0]| = {np.abs(W[0,0])}")
    if not np.allclose(W[1:, 0], 0, atol=tol):
        raise RuntimeError(f"Failed to zero first column")
    if not np.allclose(W[0, 1:], 0, atol=tol):
        raise RuntimeError(f"First row not zeroed")
    
    # Handle potential phase on W[0,0] - it should be 1, but might be e^{iφ}
    # Since W ∈ SU(d), if first row and column are [1,0,...] and [1,0,...]^T,
    # then W[0,0] must be 1 (not just |W[0,0]|=1)
    phase = W[0, 0]
    if not np.isclose(phase, 1.0, atol=tol):
        # This shouldn't happen for SU(d), but handle it
        print(f"Warning: W[0,0] = {phase}, expected 1.0")
    
    # Extract the (d-1)×(d-1) block
    W_sub = W[1:, 1:].copy()
    
    # Verify W_sub is in SU(d-1)
    det_W_sub = np.linalg.det(W_sub)
    if not np.isclose(det_W_sub, 1.0, atol=tol):
        # The determinant might be off by the phase we extracted
        # For proper SU(d), this should be 1
        print(f"Warning: det(W_sub) = {det_W_sub}")
    
    # Recursively decompose W_sub
    sub_gates = sud_decompose(W_sub, tol=tol)
    
    # Embed sub_gates into d-dimensional space (shift indices by 1)
    embedded_sub_gates: List[TwoLevelUnitary] = []
    for gate in sub_gates:
        # skip identity gates
        if np.allclose(gate.submatrix, np.eye(2), atol=tol):
            continue
        
        embedded = TwoLevelUnitary(
            dim=d,
            i=gate.i + 1,
            j=gate.j + 1,
            submatrix=gate.submatrix.copy()
        )
        embedded_sub_gates.append(embedded)
    
    if not embedded_sub_gates:
        print("Warning: No non-trivial sub-gates extracted from W_sub")
    # Reconstruct U:
    # We applied: V_{d-1} @ ... @ V_1 @ U = 1 ⊕ W_sub
    # where V_k = V_{0,k+1} in 0-indexed notation
    #
    # So: U = V_1† @ V_2† @ ... @ V_{d-1}† @ (1 ⊕ W_sub)
    #
    # v_gates = [V_1, V_2, ..., V_{d-1}] in order of application
    # We need [V_1†, V_2†, ..., V_{d-1}†, sub_gates...]
    
    result_gates: List[TwoLevelUnitary] = []
    
    # V gates need to be inverted (daggered)
    # The order: we applied V_{d-1} @ ... @ V_1 @ U
    # So U = V_1† @ V_2† @ ... @ V_{d-1}† @ (1 ⊕ W)
    for V_gate in v_gates:
        result_gates.append(V_gate.dagger())
    
    # Add the embedded sub-gates
    result_gates.extend(embedded_sub_gates)
    
    # assert that the prodcut of the result_gates equals U
    assert verify_sud_decomposition(U, result_gates), "Decomposition verification failed"
    
    return result_gates


def verify_sud_decomposition(U: np.ndarray, gates: List[TwoLevelUnitary], tol: float = 1e-10) -> bool:
    """Verify that the product of gates equals U."""
    d = U.shape[0]
    product = np.eye(d, dtype=complex)
    
    for gate in gates:
        product = product @ gate.to_full_matrix()
    
    if np.allclose(product, U, atol=tol):
        return True
    else:
        print(f"Max error: {np.max(np.abs(product - U))}")
        with np.printoptions(precision=3, suppress=True):
            print("Expected U:")
            print(U)
            print("Got product:")
            print(product)
        return False


def test_sud_decompose():
    """Test SU(d) decomposition."""
    np.random.seed(42)
    
    for d in [2, 3, 4, 5, 6]:
        print(f"\n{'='*50}")
        print(f"Testing SU({d}) decomposition")
        print(f"{'='*50}")
        
        # Generate random SU(d) matrix
        H = np.random.randn(d, d) + 1j * np.random.randn(d, d)
        Q, R = np.linalg.qr(H)
        det_Q = np.linalg.det(Q)
        U = Q / (det_Q ** (1/d))
        
        print(f"det(U) = {np.linalg.det(U):.10f}")
        
        # Decompose
        gates = sud_decompose(U)
        
        print(f"Number of 2-level gates: {len(gates)}")
        print(f"Expected: d(d-1)/2 = {d*(d-1)//2}")
        
        # Verify
        success = verify_sud_decomposition(U, gates)
        print(f"Verification: {'PASSED' if success else 'FAILED'}")


test_sud_decompose()


Testing SU(2) decomposition
det(U) = 1.0000000000-0.0000000000j
Number of 2-level gates: 1
Expected: d(d-1)/2 = 1
Verification: PASSED

Testing SU(3) decomposition
det(U) = 1.0000000000+0.0000000000j
Number of 2-level gates: 3
Expected: d(d-1)/2 = 3
Verification: PASSED

Testing SU(4) decomposition
det(U) = 1.0000000000+0.0000000000j
Number of 2-level gates: 6
Expected: d(d-1)/2 = 6
Verification: PASSED

Testing SU(5) decomposition
det(U) = 1.0000000000-0.0000000000j
Number of 2-level gates: 10
Expected: d(d-1)/2 = 10
Verification: PASSED

Testing SU(6) decomposition
det(U) = 1.0000000000+0.0000000000j
Number of 2-level gates: 15
Expected: d(d-1)/2 = 15
Verification: PASSED


In [23]:

from qiskit import QuantumCircuit
from typing import List, Tuple
import numpy as np
import scipy.linalg
from qiskit.circuit import AncillaRegister
from qiskit.quantum_info import Operator
from typing import Dict

def get_effective_unitary(gate, ancilla_indices, ancilla_state=0):
    """
    Extract the effective unitary on logical qubits when ancilla qubits
    are fixed to a specific state (default |0⟩).
    
    Args:
        gate: The full gate/circuit
        ancilla_indices: List of qubit indices that are ancillas (0-indexed)
        ancilla_state: The fixed state of ancillas (usually 0)
    
    Returns:
        The effective unitary on the logical qubits
    """
    if not isinstance(ancilla_indices, list):
        ancilla_indices = [ancilla_indices]
    
    # if gate is not a unitary matrix, convert it
    full_unitary = Operator(gate).data
    n_qubits = int(np.log2(full_unitary.shape[0]))
    
    # Find which basis states have ancilla in the specified state
    logical_indices = []
    for i in range(2**n_qubits):
        # Check if all ancilla qubits are in the correct state
        ancilla_match = all(
            ((i >> idx) & 1) == ancilla_state 
            for idx in ancilla_indices
        )
        if ancilla_match:
            logical_indices.append(i)
    
    # Extract the submatrix
    effective_U = full_unitary[np.ix_(logical_indices, logical_indices)]
    return effective_U


def apply_controlled_iswap(qc: QuantumCircuit, control: int, target1: int, target2: int, 
                        control_val: int = 1, inverse: bool = False):
    """Apply a controlled-iSWAP or controlled-iSWAP† gate.
    
    Args:
        qc: QuantumCircuit
        control: Control qubit index
        target1: First target qubit
        target2: Second target qubit
        control_val: Control value (0 or 1)
        inverse: If True, apply iSWAP† (controlled-iSWAP†)
    """
    from qiskit.circuit.library import iSwapGate
    
    # If control_val is 0, flip the control qubit
    if control_val == 0:
        qc.x(control)
    
    # Create controlled-iSWAP
    if inverse:
        gate = iSwapGate().inverse().control(1)
    else:
        gate = iSwapGate().control(1)
    
    qc.append(gate, [control, target1, target2])
    
    if control_val == 0:
        qc.x(control)

def construct_conjugation_gates(b: int, b_prime: int, n_qubits: int) -> Tuple[List[Tuple[int, int, int]], int]:
    """Construct the sequence of controlled-iSWAP gates K that conjugates
    a 2-level unitary from (b, b') to (b'', b') where d(b'', b') = 2.
    
    Following the construction in Lemma 10 of arXiv:2309.11051.
    
    Args:
        b: First basis state index
        b_prime: Second basis state index  
        n_qubits: Number of qubits
        
    Returns:
        Tuple of (gates_list, b_double_prime) where:
        - gates_list: List of (control_bit, target_bit1, target_bit2) for controlled-iSWAPs
        - b_double_prime: The intermediate state index with d(b_double_prime, b_prime) = 2
    """
    # Work with bit indices consistently using bit-indexing (LSB = bit 0)
    # Extract bits using bit operations, not string operations
    
    # Find positions where b and b' differ (using bit indexing)
    diff_positions = []
    for bit_idx in range(n_qubits):
        bit_b = (b >> bit_idx) & 1
        bit_b_prime = (b_prime >> bit_idx) & 1
        if bit_b != bit_b_prime:
            diff_positions.append(bit_idx)
    
    hamming_dist = len(diff_positions)
    
    if hamming_dist % 2 != 0:
        raise ValueError("Hamming distance must be even for equal Hamming weight states")
    
    t = hamming_dist // 2  # Number of swaps needed
    
    if t <= 1:
        # Already at distance 2 or 0
        return [], b
    
    # Partition differing positions into l_j (1 in b, 0 in b') and r_j (0 in b, 1 in b')
    l_positions = []  # positions where b has 1 and b' has 0
    r_positions = []  # positions where b has 0 and b' has 1
    
    for bit_idx in diff_positions:
        bit_b = (b >> bit_idx) & 1
        bit_b_prime = (b_prime >> bit_idx) & 1
        if bit_b == 1 and bit_b_prime == 0:
            l_positions.append(bit_idx)
        elif bit_b == 0 and bit_b_prime == 1:
            r_positions.append(bit_idx)
    
    if len(l_positions) != t or len(r_positions) != t:
        raise ValueError(f"Partition error: found {len(l_positions)} l's and {len(r_positions)} r's, expected {t} each")
    
    # Helper function to format state as binary string for display
    def format_state(state_int):
        return bin(state_int)[2:].zfill(n_qubits)
    
    print(f"    Constructing conjugation sequence:")
    print(f"      b  = |{format_state(b)}⟩ = {b}")
    print(f"      b' = |{format_state(b_prime)}⟩ = {b_prime}")
    print(f"      t = {t} (need {t-1} controlled-iSWAPs)")
    print(f"      l_positions (1→0): {l_positions}")
    print(f"      r_positions (0→1): {r_positions}")
    
    # Build sequence of controlled-iSWAP gates
    # Following Eq. (69)-(72) from the paper
    gates = []
    current_state = b  # Start from b
    
    for j in range(t - 1):  # j = 0, 1, ..., t-2 (we need t-1 gates)
        # print(f"      Step {j+1}/{t-1}: Current state: |{format_state(current_state)}⟩")
        lj = l_positions[j]
        rj = r_positions[j]
        
        # Control bit: l_{j+1}
        # This must be a bit where:
        # - current state has 1 (so gate acts)
        # - b' has 0 (so gate doesn't act on b')
        control_bit = l_positions[j + 1]
        
        print(f"      Gate {j}: Controlled-iSWAP with control={control_bit}, targets=({lj},{rj})")
        print(f"        Current state: |{format_state(current_state)}⟩")
        
        # Verify control bit is 1 in current state and 0 in b'
        current_control_bit = (current_state >> control_bit) & 1
        bprime_control_bit = (b_prime >> control_bit) & 1
        
        print(f"        Control bit {control_bit}: current={current_control_bit}, b'={bprime_control_bit}")
        
        if current_control_bit != 1:
            raise ValueError(f"Control bit {control_bit} should be 1 in current state")
        if bprime_control_bit != 0:
            raise ValueError(f"Control bit {control_bit} should be 0 in b' for gate to not act on b'")
        
        gates.append((control_bit, lj, rj))
        
        # Update current state by swapping bits at lj and rj
        # Extract the two bits
        bit_lj = (current_state >> lj) & 1
        bit_rj = (current_state >> rj) & 1
        
        # Swap them
        if bit_lj != bit_rj:  # Only need to swap if they're different
            # Clear both bits
            current_state &= ~(1 << lj)
            current_state &= ~(1 << rj)
            # Set them to swapped values
            current_state |= (bit_rj << lj)
            current_state |= (bit_lj << rj)
    
    b_double_prime = current_state
    
    print(f"      Final state b'': |{format_state(b_double_prime)}⟩ = {b_double_prime}")
    
    # Verify the Hamming distance
    hd_final = bin(b_double_prime ^ b_prime).count('1')
    print(f"      Hamming distance d(b'', b'): {hd_final}")
    
    if hd_final != 2:
        raise ValueError(f"Final Hamming distance should be 2, got {hd_final}")
    
    return gates, b_double_prime

def fix_relative_phases(qc: QuantumCircuit, qargs: List[int], phases: List[float]):
    """Fix relative phases between different Hamming weight sectors.
    
    Args:
        qc: Quantum circuit
        qargs: List of logical qubit indices
        phases: List of phases [theta_0, theta_1, ...] for each HW sector
    """
    theta_0 = phases[0]
    num_qubits = len(qargs)
    print(f"\nFixing relative phases: {phases}")
    
    # Track if we've added an ancilla (do this OUTSIDE the loop)
    anc = None
    
    for m, theta in enumerate(phases[1:], start=1):  # start=1 to get correct HW
        delta = theta - theta_0
        
        if np.abs(delta) < 1e-10:
            continue  # Skip if no phase correction needed
        print(f"\nFixing relative phase for Hamming weight {m}: delta = {delta:.4f} rad")
        # Add ancilla only once, when first needed
        if anc is None:
            if qc.num_ancillas > 0:
                # Ancilla already exists - get its index
                # The ancilla qubit index in the circuit
                anc = qc.num_qubits - qc.num_ancillas  # First ancilla index
            else:
                # Add new ancilla register
                qc.add_register(AncillaRegister(1, 'ancilla'))
                anc = qc.num_qubits - 1  # Last qubit is the ancilla
            
            # Add ancilla to qargs for the decomposition
            qargs_with_anc = qargs + [anc]
        else:
            qargs_with_anc = qargs + [anc]
        
        # Step 1: Pick |b>: any bitstring of weight m (not m+1, since enumerate starts at 1)
        b = None
        for i in range(2**num_qubits):
            bits = bin(i)[2:].zfill(num_qubits)
            if bits.count('1') == m:
                b = bits
                break

        if b is None:
            raise ValueError(f"No basis state with Hamming weight {m}")

        # Step 2: Construct |b'> by flipping one '1' → '0'
        b_list = list(b)
        flip_index = None
        for idx in range(num_qubits):
            if b_list[idx] == '1':
                b_list[idx] = '0'
                flip_index = idx
                break

        b_prime = ''.join(b_list)
        
        print(f" |b> = |{b}>, |b'> = |{b_prime}>")

        # Step 3: Embed into (n+1)-qubit Hilbert space
        # Convention: ancilla is the MOST significant bit (leftmost)
        dim = 2**(num_qubits + 1)
        H = np.zeros((dim, dim), dtype=complex)

        # |b>|0>_anc : ancilla=0 means MSB=0
        # Index = 0 * 2^n + int(b, 2) = int("0" + b, 2)
        idx_b0 = int(b, 2)  # ancilla=0, so just the system index
        
        # |b'>|1>_anc : ancilla=1 means MSB=1  
        # Index = 1 * 2^n + int(b', 2) = int("1" + b', 2) = 2^n + int(b', 2)
        idx_bp1 = (1 << num_qubits) + int(b_prime, 2)

        H[idx_b0, idx_b0] = +1   # |b>|0> term
        H[idx_bp1, idx_bp1] = -1 # |b'>|1> term
        
        with np.printoptions(precision=3, suppress=True):
            print("Applying phase correction:")
            print(f"Delta for HW {m}: {delta:.4f} rad")
            print(f"Indices: |b>|0> = {idx_b0}, |b'>|1> = {idx_bp1}")
            print("Hamming weight matrix H:")
            print_nontrivial_unitary_basis_action(H, num_qubits + 1, is_hamiltonian=True)
        
        # Exponentiate
        U_phase = scipy.linalg.expm(-1j * delta * H)
        
        with np.printoptions(precision=3, suppress=True, linewidth=1000):
            print("Submatrix to implement:")
            print_nontrivial_unitary_basis_action(U_phase, num_qubits + 1)
            print("Determinant:", np.linalg.det(U_phase))
        
        # Apply the 2-level unitary decomposition
        NQubitDecomposer.apply_2level_nqubit_hw2_unitary(
            qc, U_phase, qargs_with_anc
        )

def print_nontrivial_unitary_basis_action(U, n_qubits, is_hamiltonian=False, tol=1e-10):
    """
    Print the nontrivial actions of a unitary U on n_qubits in the
    computational basis.

    For each basis state |x>, this prints |x> -> sum_j a_j |j>
    only if U|x> is not (within tol) equal to |x>.
    """
    # dim = 2**n_qubits
    U = np.asarray(U, dtype=complex)
    dim = U.shape[0]
    n_qubits = int(np.log2(dim))
    
    if U.shape != (dim, dim):
        raise ValueError(f"U must have shape {(dim, dim)}, got {U.shape}")

    for col_idx in range(dim):
        # Column col_idx is U |col_idx>
        col = U[:, col_idx]

        # Identity action would be exactly the basis vector e_{col_idx}
        e = np.zeros(dim, dtype=complex)
        if is_hamiltonian:
            e[col_idx] = 0.0
        else:
            e[col_idx] = 1.0

        if np.allclose(col, e, atol=tol):
            # Acts as identity on |col_idx>, skip
            continue

        in_state = format(col_idx, f"0{n_qubits}b")

        # Build a readable expression for the output state
        terms = []
        for row_idx, amp in enumerate(col):
            if abs(amp) > tol:
                out_state = format(row_idx, f"0{n_qubits}b")
                # Nice-ish formatting of complex amplitudes
                if abs(amp.imag) < tol:
                    terms.append(f"{amp.real:+.3f}|{out_state}>")
                elif abs(amp.real) < tol:
                    terms.append(f"{amp.imag:+.3f}j|{out_state}>")
                else:
                    terms.append(f"({amp.real:+.3f}{amp.imag:+.3f}j)|{out_state}>")

        rhs = " + ".join(terms) if terms else "0"
        print(f"|{in_state}> -> {rhs}")

def su2_AB_from_U(U: np.ndarray, atol: float = 1e-10) -> Tuple[np.ndarray, np.ndarray]:
    """Find A,B ∈ SU(2) such that A B A† B† = U.
    
    This construction follows Eq. (64) from the paper.
    
    Args:
        U: 2x2 unitary matrix
        atol: Absolute tolerance for numerical comparisons
        
    Returns:
        Tuple of (A, B) matrices in SU(2)
        
    Raises:
        ValueError: If U is not 2x2 or has zero determinant
    """
    if U.shape != (2, 2):
        raise ValueError("U must be 2x2")

    # Remove global phase so that det(U_su2) = 1
    detU = np.linalg.det(U)
    # Check that detU is 1 
    if not np.isclose(abs(detU), 1.0, atol=atol):
        raise ValueError(f"Determinant of U must have magnitude 1, got {detU}")
    
    U_su2 = U 

    evals, evecs = np.linalg.eig(U_su2)
    φ1, φ2 = np.angle(evals[0]), np.angle(evals[1])

    θ = 0.5 * (φ1 - φ2)

    # Build exp(i θ Z/2) and exp(i θ Z)
    # Z = np.diag([1.0, -1.0])
    D = np.diag([np.exp(1j * θ), np.exp(-1j * θ)])
    
    W = evecs  # columns are eigenvectors
    
    if not np.allclose(W @ D @ W.conj().T, U_su2, atol=1e-6):
        # Eigenvalues are in opposite order, swap columns of W
        W = W[:, ::-1]
        # Or equivalently, negate θ
        # θ = -θ
        if not np.allclose(W @ D @ W.conj().T, U_su2, atol=1e-6):
            raise ValueError("Eigen-decomposition failed to reconstruct U_su2")
        
    exp_iθZ_over2 = np.diag(np.exp(1j * θ * np.array([1.0, -1.0]) / 2.0))
    # A(1) = W exp(i θ Z/2) W†
    A1 = W @ exp_iθZ_over2 @ W.conj().T

    # B(1) = i W X W†, where X = [[0,1],[1,0]]
    X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
    B1 = 1j * W @ X @ W.conj().T

    return A1, B1


In [ ]:

from qiskit.transpiler import TransformationPass, Target
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit
from qiskit.converters import dag_to_circuit, circuit_to_dag
from qiskit.quantum_info import Operator
from qiskit.synthesis import OneQubitEulerDecomposer
from reck_decompose import reck_decomposition 
from utils import TwoLevelBlock
  
from qiskit.circuit.library import UnitaryGate, GlobalPhaseGate

from decomposer import GateSynthesizer, TwoQubitDecomposer, ThreeQubitDecomposer, NQubitDecomposer, PhaseCorrector

from typing import List, Dict, Tuple
import numpy as np
from utils import is_energy_conserving, get_hamming_weight_blocks, extract_two_level_blocks, get_nontrivial_rows_cols
from utils import ControlledTwoLevel


class NQubitDecomposer:
    """Decomposes n-qubit energy-conserving unitaries."""
    
    @staticmethod
    def decompose_ncontrolled_2level_unitary(U: np.ndarray, control_bits: List[int], control_vals: List[int], target_bits: List[int]) -> List[ControlledTwoLevel]:
        k = len(control_bits)

        # Base case: single control bit
        if k == 1:
            return [ControlledTwoLevel(U=U, control_bits=control_bits.copy(),
                                      control_vals=control_vals.copy(),
                                      target_bits=target_bits.copy())]
        

        detU = np.linalg.det(U)
        U_su2 = U

        # Find A(1), B(1) ∈ SU(2) s.t. A B A† B† = U_su2
        A1, B1 = su2_AB_from_U(U_su2)
        
        # assert if AA† = I and BB† = I
        assert np.allclose(A1 @ A1.conj().T, np.eye(2)), "A1 is not unitary"
        assert np.allclose(B1 @ B1.conj().T, np.eye(2)), "B1 is not unitary"
        
        assert np.allclose(A1 @ B1 @ A1.conj().T @ B1.conj().T, U_su2), f"SU(2) decomposition failed for {U_su2}"
                

        # Split control bits into two halves
        mid = k // 2
        c1 = control_bits[:mid]
        v1 = control_vals[:mid]
        c2 = control_bits[mid:]
        v2 = control_vals[mid:]

        # Recursive decomposition: Λ_c(U) = Λ_{c1}(A) Λ_{c2}(B) Λ_{c1}(A†) Λ_{c2}(B†)
        gates = []
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(A1, c1, v1, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(B1, c2, v2, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(A1.conj().T, c1, v1, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(B1.conj().T, c2, v2, target_bits)
        

        return gates
    
    @staticmethod
    def apply_2level_nqubit_hw2_unitary(qc: QuantumCircuit, U: np.ndarray, 
                                   qargs: List[int], hw_indices: List[int] = None):
        """Apply a 2-level unitary embedded in n-qubits.

        Args:
            qc: QuantumCircuit to append gates to
            U: 2^n x 2^n unitary matrix (2-level in computational basis)
            qargs: list of n physical qubit indices
            hw_indices: Optional list of basis indices in the current HW block
        """
        n = qc.num_qubits
        

        # Get the two basis states that are coupled
        local_i, local_j = get_nontrivial_rows_cols(U)
        
        if local_j < local_i:
            local_i, local_j = local_j, local_i  # swap to ensure i < j
        
        # Extract the 2x2 submatrix
        submatrix = np.array([
            [U[local_i, local_i], U[local_i, local_j]],
            [U[local_j, local_i], U[local_j, local_j]]
        ], dtype=complex)
        
        # submatrix = -1 * submatrix  # Adjust global phase for consistency
        
            # Make it SU(2): factor out determinant as a scalar phase
        detU = np.linalg.det(submatrix)
        if np.abs(detU) < 1e-12:
            raise ValueError("2x2 submatrix determinant is ~0; not unitary?")
        su2_block = submatrix   # scalar goes into D, not Q
        
        # with np.printoptions(precision=3, suppress=True, linewidth=120):
        #     print(f"  Extracted 2-level SU(2) block (det {detU:.3f}):\n {su2_block}")
        

        # Determine control bits and target bits
        bits_i = bin(local_i)[2:].zfill(n)
        bits_j = bin(local_j)[2:].zfill(n)
        print(f"  Applying TLU: |{bits_i}⟩ ↔ |{bits_j}⟩ on local indices {local_i}, {local_j}")
        hd = sum(b1 != b2 for b1, b2 in zip(bits_i, bits_j))
        print(f"  [apply_2level_nqubit_hw2_unitary] local_i={local_i} |{bits_i}⟩, local_j={local_j} |{bits_j}⟩, HD={hd}")
    

        control_bits = []
        control_vals = []
        target_bits = []
        
        for bit in range(n):
            bit_i = (local_i >> bit) & 1
            bit_j = (local_j >> bit) & 1
            if bit_i == bit_j:
                control_bits.append(bit)
                control_vals.append(bit_i)
            else:
                target_bits.append(bit)
        
        print(f"    Target bits differ at positions: {target_bits} and control bits: {control_bits} with values {control_vals}")

        if len(target_bits) != 2:
            raise ValueError(f"2-level unitary does not differ in exactly 2 bit positions. target_bits={target_bits}, control_bits={control_bits}")

        print(f"    Controls: {list(zip(control_bits, control_vals))}, Targets: {target_bits}")

        # Decompose into single-controlled gates
        controlled_blocks = NQubitDecomposer.decompose_ncontrolled_2level_unitary(
            su2_block, control_bits, control_vals, target_bits
        )
        
        #         # Helper: map pattern on two targets to basis index (order: [t0, t1])
        def basis_index_on_targets(pat):
            return (int(pat[0]) << 1) + int(pat[1])

        # Implement each controlled block
        for block in controlled_blocks:
            
            # with np.printoptions(precision=3, suppress=True):
            #     # print controlled block info, its determinant
            #     print(f"  Controlled block U:\n {block.U}")
            #     print(f"  Determinant: {np.linalg.det(block.U):.3f}")
            
            theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
                Operator(block.U)
            )

            detU = np.linalg.det(block.U)

            # Normalize phase into (-π, π]
            phase = (global_phase + np.pi) % (2 * np.pi) - np.pi

            if np.isclose(detU, 1.0, atol=1e-8):
                if np.isclose(phase, 0.0, atol=1e-8):
                    # Already SU(2) up to numerical noise
                    print(f"    Det≈1, phase≈0; θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}")
                elif np.isclose(abs(phase), np.pi, atol=1e-8):
                    # U is - (Rx Ry Rx); absorb -1 into one rotation using 2π periodicity
                    print("    Det≈1 but global phase ≈ π; absorbing into θ.")
                    theta += 2 * np.pi          # could also adjust φ or λ instead
                    global_phase = 0.0
                    print(f"    Adjusted angles: θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}, phase={global_phase:.3f}")
                else:
                    # In exact math this shouldn't happen if det(U)=1; probably numerical issues
                    print(f"    Warning: det≈1 but unexpected global phase {phase:.6f}")
            else:
                print(f"    det(U) ≈ {detU}, leaving global phase as given: {global_phase:.6f}")


            if len(block.control_bits) != 1:
                raise RuntimeError(
                    "Final decomposition should only contain single-controlled gates."
                )

            ctrl_bit = block.control_bits[0]
            ctrl_val = block.control_vals[0]
            t0_bit, t1_bit = block.target_bits

            control = qargs[ctrl_bit]
            target1 = qargs[t0_bit]
            target2 = qargs[t1_bit]

            print(
                f"    Implementing single-controlled SU(2) on targets {block.target_bits} "
                f"with control bit {ctrl_bit}={ctrl_val}, angles XYX:"
                f" θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}, phase={global_phase:.3f}"
            )

            # phase_correction = np.exp(1j * global_phase)
            # # Build the controlled gate on the two targets
            # phase_gate = np.diag([ phase_correction, phase_correction,
            #                       phase_correction, phase_correction])
            # qc.append(UnitaryGate(block.U, label="ctl-2lvl"), [control, target1, target2])
            # Apply U = Rx(θ) Ry(φ) Rx(λ)
            GateSynthesizer.controlled_exp_i_alpha_R(qc, -phi/2, ctrl_val, target1, target2, control)
            GateSynthesizer.controlled_exp_i_theta_L(qc, -theta/2, ctrl_val, target1, target2, control)
            GateSynthesizer.controlled_exp_i_alpha_R(qc, -lam/2, ctrl_val, target1, target2, control)
            # qc.append(UnitaryGate(phase_gate, label="global_phase").control(1, ctrl_state=str(ctrl_val)), [control, target1, target2])  # Global phase gate
      
         
    @staticmethod
    def decompose_general_unitary(qc: QuantumCircuit, U: np.ndarray, qargs: List[int]) -> Dict[int, List[float]]:
        """Decompose and apply a general energy-conserving unitary on n qubits.

        Args:
            qc: QuantumCircuit to append gates to
            U: unitary matrix (2^n x 2^n)
            qargs: [q0, q1, q2, ...] physical qubit indices
            
        Returns:
            Dictionary mapping Hamming weight to list of phases
        """
        n_qubits = len(qargs)
        dim = 2**n_qubits

        if U.shape != (dim, dim):
            raise ValueError(
                f"Unitary shape {U.shape} incompatible with {n_qubits} qubits."
            )
        # conirm unitarity
        if not np.allclose(U.conj().T @ U, np.eye(dim), atol=1e-10):
            raise ValueError("Input matrix U is not unitary.")

        if not is_energy_conserving(U):
            raise ValueError("Unitary is not energy-conserving")

        # Group basis states by Hamming weight
        hw_to_indices: Dict[int, List[int]] = {}
        for i in range(dim):
            hw = bin(i).count("1")
            hw_to_indices.setdefault(hw, []).append(i)

        hw_phases: Dict[int, List[float]] = {}
        
        blocks = get_hamming_weight_blocks(U)

        for hw in sorted(hw_to_indices.keys()):
            indices = hw_to_indices[hw]
            block_size = len(indices)

            block = blocks[hw]

            if block_size == 1:
                phase = np.angle(block[0, 0])
                hw_phases[hw] = phase
                continue
                        
            det_block = np.linalg.det(block)
            su2_block = block / np.pow(det_block, 1/block.shape[0])
            phase = np.angle(det_block) / block.shape[0]
            print(f"\nDecomposing Hamming weight {hw} block (size {block_size}) [{block.shape}]: Phase in degrees: {phase * 180 / np.pi:.3f}")  
            # phase = np.angle(np.pow(det_block, 1/2))
        
            hw_phases[hw] = phase * 0   # Store negative phase for correction later
            
            # confirm that that su2_block has determinant 1
            if not np.isclose(np.linalg.det(su2_block), 1.0, atol=1e-10):
                raise ValueError("Normalized block does not have determinant 1")
            block = su2_block
            
            two_level_blocks = extract_two_level_blocks(block, atol=1e-10, enforce_hamming_distance=None)
            
            with np.printoptions(precision=3, suppress=True):
                # print block 
                print(f"  Extracted SU({block.shape[0]}) block:\n{block}")
                # print two-level blocks
                print(f"  Extracted {len(two_level_blocks)} two-level blocks:")
            
            
            if not (two_level_blocks or np.allclose(block, np.eye(block_size), atol=1e-10)):    
                # decompose the block using Reck decomposition as fallback
                print("  Warning: Could not fully decompose block into 2-level unitaries.")
                print("  Falling back to Reck decomposition.")
                # reconstruct full unitary from block
                # full_block = np.eye(dim, dtype=complex)
                # for ii, global_i in enumerate(indices):
                #     for jj, global_j in enumerate(indices):
                #         full_block[global_i, global_j] = block[ii, jj]
                # with np.printoptions(precision=3, suppress=True, linewidth=120):
                #     print("  Full block for Reck decomposition:")
                #     print(full_block)
                
                Us = sud_decompose(block)
                two_level_blocks = []
                print("  Extracted TLU from Reck decomposition:")
                with np.printoptions(precision=3, suppress=True):
                    print(f"  Reck decomposition yielded {len(Us)} two-level blocks:")
                    for tlu in Us:
                        print(tlu.submatrix)
                        two_level_blocks.append(
                            tlu
                        )
                # reverse the order to match application order
                # two_level_blocks = list(reversed(two_level_blocks))
                
            for tlb in two_level_blocks:                
                with np.printoptions(precision=3, suppress=True):
                    print(f"  TLU between local indices {tlb.i}, {tlb.j} with submatrix:\n{tlb.submatrix}")
                    print(f" Determinant: {np.linalg.det(tlb.submatrix):.3f}")
                    
                if np.allclose(tlb.submatrix, np.eye(2), atol=1e-10):
                    print("    Skipping identity TLU.")
                    continue
                
                 # Compute Hamming distance between basis states |i⟩ and |j⟩
                
                
                bits_i = bin(indices[tlb.i])[2:].zfill(n_qubits)
                bits_j = bin(indices[tlb.j])[2:].zfill(n_qubits)
                hd = sum(b1 != b2 for b1, b2 in zip(bits_i, bits_j))
                print(f"    Hamming distance between |{bits_i}⟩ and |{bits_j}⟩: {hd}")
                
                global_i = indices[tlb.i]
                global_j = indices[tlb.j]
                
                print(f" Global indices: {global_i}, {global_j}")
                print(f" Local indices in block: {tlb.i}, {tlb.j}")
                print(f"SUbmatrix:\n{tlb.submatrix}")
                
                
                if hd == 2:
                    full_U = np.eye(dim, dtype=complex)
                    full_U[global_i, global_i] = tlb.submatrix[0, 0]
                    full_U[global_i, global_j] = tlb.submatrix[0, 1]
                    full_U[global_j, global_i] = tlb.submatrix[1, 0]
                    # full_U[global_i, global_j] = tlb.submatrix[1, 0]
                    # full_U[global_j, global_i] = tlb.submatrix[0, 1]
                    full_U[global_j, global_j] = tlb.submatrix[1, 1]
                    NQubitDecomposer.apply_2level_nqubit_hw2_unitary(qc, full_U, qargs, indices)
                    
                elif hd > 2:
                    print(f"    Using conjugation method (Hamming distance = {hd})")
                    
                    conj_gates, b_double_prime = construct_conjugation_gates(
                        global_i, global_j, n_qubits
                    )
                    

                    for ctrl, t1, t2 in conj_gates:
                        apply_controlled_iswap(qc, qargs[ctrl], qargs[t1], qargs[t2])
                    
                    # Construct W = K V K† acting on (b'', b')
                    # According to Eq. (76) of the paper
                    t = hd // 2  # Total number of swaps needed
                    
                    # Phase factors from Eq. (76): ̃U = diag(i^(t-1), 1) · U · diag(i^(-(t-1)), 1)
                    phase_factor_upper = (1j) ** (t - 1)      # i^(t-1)
                    phase_factor_lower = (1j) ** (1 - t)      # i^(1-t)
                    
                    
                    W_matrix = np.array([
                        [tlb.submatrix[0, 0], phase_factor_upper * tlb.submatrix[0, 1]],
                        [phase_factor_lower * tlb.submatrix[1, 0], tlb.submatrix[1, 1]]
                    ], dtype=complex)
                    
                    print(f"    Constructed W matrix with phase factors (t={t}):\n{W_matrix}")
                                        
                    
                    # Apply W (which has Hamming distance 2)
                    full_W = np.eye(dim, dtype=complex)
                    full_W[b_double_prime, b_double_prime] = W_matrix[0, 0]
                    full_W[b_double_prime, global_j] = W_matrix[0, 1]
                    full_W[global_j, b_double_prime] = W_matrix[1, 0]
                    # full_W[b_double_prime, global_j] = W_matrix[ 1, 0]
                    # full_W[global_j, b_double_prime] = W_matrix[0, 1]
                    full_W[global_j, global_j] = W_matrix[1, 1]
                    
                    NQubitDecomposer.apply_2level_nqubit_hw2_unitary(qc, full_W, qargs, indices)
                    
                    for ctrl, t1, t2 in reversed(conj_gates):
                        apply_controlled_iswap(qc, qargs[ctrl], qargs[t1], qargs[t2], inverse=True)
                        
                else:
                    print(f"    Skipping: Hamming distance = {hd}")
                    continue
        
        print("\nFixing relative phases between Hamming weight sectors...")
        print(f"  HW phases: {hw_phases}")

        fix_relative_phases(
            qc, qargs,
            [hw_phases.get(i, 0.0) for i in range(n_qubits + 1)]
        )       
        return hw_phases



class EnergyConservationDecompositionPass(TransformationPass):
    """A custom transpiler pass that decomposes energy-conserving unitaries
    into a basis set of gates that also conserve energy.
    """

    def __init__(self, target: Target):
        """Initialize the pass.
        
        Args:
            target: Target backend specification
        """
        super().__init__()
        self.target = target
    
    def run(self, dag: DAGCircuit) -> DAGCircuit:
        """Run the pass on the given DAGCircuit.

        Args:
            dag: The input DAGCircuit to be transformed.

        Returns:
            The transformed DAGCircuit with decomposed gates.
        """
        circuit = dag_to_circuit(dag)
        
        anc = QuantumRegister(1, 'anc')
        qregs = dag.qregs
        cregs = dag.cregs
        
        new_circuit = QuantumCircuit(*qregs.values(), anc, *cregs.values())
        
        for instr, qargs, cargs in circuit.data:
            qindices = [q._index for q in qargs]
            
            if instr.name == 'cz':
                q0, q1 = qindices[0], qindices[1]
                GateSynthesizer.apply_cz_decomposition(
                    new_circuit, q0, q1, dag.num_qubits()
                )
            
            elif instr.name == 'swap':
                q0, q1 = qindices[0], qindices[1]
                GateSynthesizer.apply_swap_decomposition(
                    new_circuit, q0, q1, dag.num_qubits()
                )
                
            elif instr.num_qubits == 2:
                phases = TwoQubitDecomposer.decompose_2qubit_ec_gate(
                    new_circuit, instr, qindices
                )
                PhaseCorrector.fix_relative_phases(
                    new_circuit, qindices, dag.num_qubits(), phases
                )
            
            elif instr.num_qubits == 3:
                U = Operator(instr).data
                phases = ThreeQubitDecomposer.decompose_3qubit_unitary(
                    new_circuit, U, qindices
                )
                PhaseCorrector.fix_relative_phases(
                    new_circuit, qindices, dag.num_qubits(), phases
                )
            
            elif instr.num_qubits > 3:
                U = Operator(instr).data
                hw_phases = NQubitDecomposer.decompose_general_unitary(
                    new_circuit, U, qindices
                )
                # Get phases as list
                phases = [hw_phases.get(i, [0.0])[0] for i in range(instr.num_qubits + 1)]
                PhaseCorrector.fix_relative_phases(
                    new_circuit, qindices, dag.num_qubits(), phases
                )
            
            else:
                # Single qubit gates or unsupported gates - pass through
                new_circuit.append(instr, qargs, cargs)
        
        new_dag = circuit_to_dag(new_circuit)
        return new_dag

In [4]:
# Test the CZ decomposition

from utils import get_effective_unitary, _unitaries_close_up_to_phase
from qiskit.circuit.library import CZGate
import numpy as np

def test_apply_cz_decomposition():
    qc = QuantumCircuit(3)
    ancilla_index = 2  # Using qubit 2 as ancilla
    pass_instance = EnergyConservationDecompositionPass(target=None)
    # pass_instance._apply_cz_decomposition(qc, 0, 1, ancilla_index)
    GateSynthesizer.apply_cz_decomposition(qc, 0, 1, 2)
    
    U_impl = Operator(qc).data
    
    effective_U = get_effective_unitary(qc, [ancilla_index], ancilla_state=0)
    
    
    # Ideal CZ gate on qubits 0 and 1
    qc_ideal = QuantumCircuit(3)
    qc_ideal.append(CZGate(), [0, 1])
    U_ideal = Operator(qc_ideal).data
    effective_U_ideal = get_effective_unitary(qc_ideal, [ancilla_index], ancilla_state=0)
    
    # # print effective unitaries
    # with np.printoptions(precision=3, suppress=True):
    #     print("Effective implemented unitary:\n", effective_U)
    #     print("Effective ideal unitary:\n", effective_U_ideal)
    
    print("||effective_U - effective_U_ideal||_max =", np.max(np.abs(effective_U - effective_U_ideal)))
    print("Test passed!" if np.allclose(effective_U, effective_U_ideal) else "Test failed!")

test_apply_cz_decomposition()

||effective_U - effective_U_ideal||_max = 2.220446049250313e-16
Test passed!


In [5]:
# Test the SWAP decomposition
def test_apply_swap_decomposition():
    qc = QuantumCircuit(3)
    ancilla_index = 2  # Using qubit 2 as ancilla
    pass_instance = EnergyConservationDecompositionPass(target=None)
    # pass_instance._apply_swap_decomposition(qc, 0, 1, ancilla_index)
    GateSynthesizer.apply_swap_decomposition(qc, 0, 1, 2)
    
    U_impl = Operator(qc).data
    
    effective_U = get_effective_unitary(qc, [ancilla_index], ancilla_state=0)
    
    
    # Ideal SWAP gate on qubits 0 and 1
    qc_ideal = QuantumCircuit(3)
    qc_ideal.swap(0, 1)
    U_ideal = Operator(qc_ideal).data
    effective_U_ideal = get_effective_unitary(qc_ideal, [ancilla_index], ancilla_state=0)
    
    # print effective unitaries
    # with np.printoptions(precision=3, suppress=True):
    #     print("Effective implemented unitary:\n", effective_U)
    #     print("Effective ideal unitary:\n", effective_U_ideal)
    
    print("||effective_U - effective_U_ideal||_max =", np.max(np.abs(effective_U - effective_U_ideal)))
    print("Test passed!" if np.allclose(effective_U, effective_U_ideal) else "Test failed!")
test_apply_swap_decomposition()

||effective_U - effective_U_ideal||_max = 2.7755575615628914e-16
Test passed!


In [6]:
# Test the exp_i_alpha_R function
def test_exp_i_alpha_R_identity_and_group():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    # --- Identity at theta = 0 ---
    qc_id = QuantumCircuit(2)
    obj.exp_i_alpha_R(qc_id, 0.0, 0, 1)
    U_id = Operator(qc_id).data
    assert np.allclose(U_id, np.eye(4, dtype=complex), atol=1e-8), \
        "exp_i_alpha_R(0) is not identity"

    # --- Group property: U(theta1+theta2) = U(theta2) U(theta1) ---
    theta1 = 0.37
    theta2 = -0.81

    # Direct: U(theta1+theta2)
    qc_direct = QuantumCircuit(2)
    obj.exp_i_alpha_R(qc_direct, theta1 + theta2, 0, 1)
    U_direct = Operator(qc_direct).data

    # Composition: U(theta2) · U(theta1)
    qc_comp = QuantumCircuit(2)
    obj.exp_i_alpha_R(qc_comp, theta1, 0, 1)
    obj.exp_i_alpha_R(qc_comp, theta2, 0, 1)
    U_comp = Operator(qc_comp).data

    assert _unitaries_close_up_to_phase(U_direct, U_comp), \
        "Group property failed for exp_i_alpha_R"
    
    print("test_exp_i_alpha_R_identity_and_group passed.")
    
# Test the controlled_exp_i_alpha_R function
def _ideal_controlled_R(theta, b, obj):
    """
    Build the ideal 3-qubit unitary for controlled exp_i_alpha_R(theta)
    with control on qubit 2, targets on (0,1), conditioned on value b.
    """
    # 2-qubit R unitary
    qc_R = QuantumCircuit(2)
    obj.exp_i_alpha_R(qc_R, theta, 0, 1)
    U_R = Operator(qc_R).data  # 4x4

    # Full 3-qubit ideal: block-diagonal
    # basis: |000>=0..|011>=3 (control=0), |100>=4..|111>=7 (control=1)
    U_ideal = np.eye(8, dtype=complex)

    if b == 0:
        # apply U_R when control qubit = 0 → upper-left 4x4 block
        U_ideal[0:4, 0:4] = U_R
        # leave lower block identity
    else:
        # apply U_R when control qubit = 1 → lower-right 4x4 block
        U_ideal[4:8, 4:8] = U_R
        # leave upper block identity

    return U_ideal

def test_controlled_exp_i_alpha_R_identity():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    qc = QuantumCircuit(3)
    obj.controlled_exp_i_alpha_R(qc, theta=0.0, b=0, q0=0, q1=1, control=2)
    U = Operator(qc).data
    assert np.allclose(U, np.eye(8, dtype=complex), atol=1e-8), \
        "controlled_exp_i_alpha_R(0) is not identity"


def test_controlled_exp_i_alpha_R_against_ideal():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    theta = 0.53

    for b in [0, 1]:
        # implemented
        qc_impl = QuantumCircuit(3)
        obj.controlled_exp_i_alpha_R(qc_impl, theta, b, 0, 1, 2)
        U_impl = Operator(qc_impl).data

        # ideal
        U_ideal = _ideal_controlled_R(theta, b, obj)

        assert _unitaries_close_up_to_phase(U_impl, U_ideal), \
            f"controlled_exp_i_alpha_R mismatch for b={b}"

def test_controlled_exp_i_alpha_R_conditioning():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    theta = 0.77
    b = 1  # test for one value; you can also loop over both

    qc_impl = QuantumCircuit(3)
    obj.controlled_exp_i_alpha_R(qc_impl, theta, b, 0, 1, 2)
    U_impl = Operator(qc_impl).data
    
    # with np.printoptions(precision=3, suppress=True, linewidth=120):
    #     print("Implemented U:\n", U_impl)
    #     # Ideal U:
    #     U_ideal = _ideal_controlled_R(theta, b, obj)
    #     print("Ideal U:\n", U_ideal)
    # states where control bit (q2) != b should be unchanged (up to phase)
    for idx in range(8):
        bits = format(idx, "03b")
        control_val = int(bits[0])  # q2 is rightmost bit
        if control_val != b:
            e = np.zeros(8, dtype=complex)
            e[idx] = 1.0
            v = U_impl @ e
            # v should be proportional to e
            nonzero = np.where(np.abs(v) > 1e-8)[0]
            assert len(nonzero) == 1 and nonzero[0] == idx, \
                "Control off-state leaked to other basis vectors"
    print("test_controlled_exp_i_alpha_R passed.")

# Test the controlled_exp_i_theta_L function
def test_controlled_exp_i_theta_L_identity_and_group():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    b = 1

    # identity check
    qc_id = QuantumCircuit(3)
    # obj.controlled_exp_i_theta_L(0.0, b, qc_id, 0, 1, 2)
    obj.controlled_exp_i_theta_L(qc_id, 0.0, b, 0, 1, 2)
    U_id = Operator(qc_id).data
    assert np.allclose(U_id, np.eye(8, dtype=complex), atol=1e-8), \
        "controlled_exp_i_theta_L(0) is not identity"

    # group property
    theta1 = 0.41
    theta2 = -0.66

    qc_direct = QuantumCircuit(3)
    obj.controlled_exp_i_theta_L(qc_direct, theta1 + theta2, b, 0, 1, 2)
    U_direct = Operator(qc_direct).data

    qc_comp = QuantumCircuit(3)
    obj.controlled_exp_i_theta_L(qc_comp, theta1, b, 0, 1, 2)
    obj.controlled_exp_i_theta_L(qc_comp, theta2, b, 0, 1, 2)
    U_comp = Operator(qc_comp).data

    assert _unitaries_close_up_to_phase(U_direct, U_comp), \
        "Group property failed for controlled_exp_i_theta_L"

def test_controlled_exp_i_theta_L_conditioning():
    # obj = EnergyConservationDecompositionPass(target=None)
    obj = GateSynthesizer()
    theta = 0.6
    b = 0

    qc_impl = QuantumCircuit(3)
    obj.controlled_exp_i_theta_L(qc_impl, theta, b, 0, 1, 2)
    U_impl = Operator(qc_impl).data

    # states where control bit (q2) != b should be eigenvectors with no leakage
    for idx in range(8):
        bits = format(idx, "03b")
        control_val = int(bits[0])  # q2 is rightmost
        if control_val != b:
            e = np.zeros(8, dtype=complex)
            e[idx] = 1.0
            v = U_impl @ e
            nonzero = np.where(np.abs(v) > 1e-8)[0]
            assert len(nonzero) == 1 and nonzero[0] == idx, \
                "controlled_exp_i_theta_L leaked when control off"

# Run the tests
test_controlled_exp_i_theta_L_identity_and_group()
test_controlled_exp_i_theta_L_conditioning()


# Run the tests
test_controlled_exp_i_alpha_R_identity()
test_controlled_exp_i_alpha_R_against_ideal()
test_controlled_exp_i_alpha_R_conditioning()

test_exp_i_alpha_R_identity_and_group()

test_controlled_exp_i_alpha_R passed.
test_exp_i_alpha_R_identity_and_group passed.


In [7]:
# Test a 3-qubit two-level unitary (8x8)
def make_3q_two_level_unitary():
    """
    Construct an 8x8 unitary that is identity except for a 2x2 SU(2) block
    mixing |001> (index 1) and |010> (index 2).

    Returns:
        U: 8x8 np.ndarray complex
    """
    # Generic SU(2) block
    theta = np.pi /8
    phi = np.pi /8

    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)

    # Example SU(2) on span{|001>, |010>}
    sub = np.array(
        [
            [c, -np.exp(1j * phi) * s],
            [np.exp(-1j * phi) * s, c],
        ],
        dtype=complex,
    )

    # Full 3-qubit unitary: identity except this 2x2 block
    U = np.eye(8, dtype=complex)

    # basis: |000>=0, |001>=1, |010>=2, |011>=3,
    #        |100>=4, |101>=5, |110>=6, |111>=7
    i = 1  # |001>
    j = 2  # |010>

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    # print the euler angles of the submatrix
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    print(f"Euler angles (XYX) of submatrix: theta={theta_x}, phi={phi_y}, lambda={lam_x}, global_phase={global_phase}")

    return U


# --------------------------------------------------------------------
# 2. Test _apply_2level_3qubit_unitary with full 8x8 U
# --------------------------------------------------------------------
def test_apply_2level_3qubit_unitary():
    # build target unitary
    # U_target = make_3q_two_level_unitary()
    n_qubits = 3
    dim = 2**n_qubits
    
    theta = np.pi / 4
    U_target = np.eye(dim, dtype=complex)
    
    # Rotate |001⟩ (idx=1) ↔ |100⟩ (idx=4)
    U_target[1, 1] = np.cos(theta)
    U_target[1, 4] = -1j * np.sin(theta)
    U_target[4, 1] = -1j * np.sin(theta)
    U_target[4, 4] = np.cos(theta)

    # new circuit
    qc = QuantumCircuit(3)

    # instantiate your pass
    # from your_module import EnergyConservationDecompositionPass
    pass_obj = ThreeQubitDecomposer()

    # physical qubit mapping
    qargs = [0, 1, 2]

    # hw_indices isn't actually used in your new implementation,
    # but we'll pass the natural mapping 0..7 for completeness.
    hw_indices = list(range(8))

    # apply your 2-level decomposition
    pass_obj.apply_2level_3qubit_unitary(qc, U_target, qargs)

    # implemented unitary
    U_impl = Operator(qc).data
    
    with np.printoptions(precision=3, suppress=True, linewidth=10000):
        print("Implemented unitary U_impl:\n", U_impl)
        print("Target unitary U_target:\n", U_target)
    
    # get the submatrix corresponding to |001> and |010>
    sub_impl = np.array(
        [
            [U_impl[1, 1], U_impl[1, 2]],
            [U_impl[2, 1], U_impl[2, 2]],
        ],
        dtype=complex,
    )
    sub_target = np.array(
        [
            [U_target[1, 1], U_target[1, 2]],
            [U_target[2, 1], U_target[2, 2]],
        ],
        dtype=complex,
    )
    xyx_decomposer = OneQubitEulerDecomposer('XYX')
    with np.printoptions(precision=3, suppress=True):
        print("Implemented submatrix:\n", sub_impl)
        print("Target submatrix:\n", sub_target)
        theta_impl, phi_impl, lam_impl, phase_impl = xyx_decomposer.angles_and_phase(
            Operator(sub_impl)
        )
        theta_target, phi_target, lam_target, phase_target = xyx_decomposer.angles_and_phase(
            Operator(sub_target)
        )
        # print(f"Implemented submatrix angles: theta={theta_impl}, phi={phi_impl}, lambda={lam_impl}, phase={phase_impl}")
        # print(f"Target submatrix angles: theta={theta_target}, phi={phi_target}, lambda={lam_target}, phase={phase_target}")
        
        

    # compare
    diff = U_impl - U_target
    max_err = np.max(np.abs(diff))
    print("Max entry-wise error ||U_impl - U_target||_max =", max_err)

    # If you want a hard test:
    # assert max_err < 1e-8, "Two-level 3-qubit decomposition failed tolerance!"
    
    # test upto global phase
    assert _unitaries_close_up_to_phase(sub_impl, sub_target), "Two-level 3-qubit decomposition failed up to global phase!"



test_apply_2level_3qubit_unitary()

  Applying TLU: |001⟩ ↔ |100⟩ on local indices 1, 4
    Control: q1=0, Targets: q[0, 2]
    XYX angles: θ=0.000, φ=2.356, λ=-0.785
Implemented unitary U_impl:
 [[ 1.   -0.j     0.   +0.j     0.   +0.j     0.   +0.j     0.   +0.j     0.   +0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j     0.707-0.j    -0.   +0.j     0.   +0.j     0.   -0.707j  0.   +0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     1.   +0.j     0.   +0.j    -0.   -0.j     0.   +0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     0.   +0.j     1.   +0.j     0.   +0.j     0.   -0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j    -0.   -0.707j -0.   +0.j     0.   +0.j     0.707+0.j     0.   +0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     0.   +0.j    -0.   -0.j     0.   +0.j     1.   -0.j     0.   +0.j     0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     0.   +0.j    -0.   +0.j     0.   +0.j     0.   -0.j     1.   -0.j     0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     0.  

In [8]:
from qiskit.circuit.library import SwapGate
# ---------------------------------------------------------
# Build exp(i θ SWAP_01 ⊗ SWAP_23)
# ---------------------------------------------------------
def make_exp_i_theta_swap12_swap34(theta):
    """
    Construct U = exp(i θ (SWAP_01 · SWAP_23)) for 4 qubits.
    Uses S^2 = I => exp(i θ S) = cos θ I + i sin θ S.
    """
    # 4-qubit SWAP_01 followed by SWAP_23
    qc_swap = QuantumCircuit(4)
    qc_swap.append(SwapGate(), [0, 1])
    qc_swap.append(SwapGate(), [2, 3])
    S = Operator(qc_swap).data  # 16x16 involution

    dim = S.shape[0]
    I = np.eye(dim, dtype=complex)

    U = np.cos(theta) * I + 1j * np.sin(theta) * S
    return U


# ---------------------------------------------------------
# The test itself
# ---------------------------------------------------------
def test_decompose_exp_i_theta_swap12_swap34():
    # choose some nontrivial angle
    theta = np.pi / 2

    # 1) Ideal target unitary
    U_target = make_exp_i_theta_swap12_swap34(theta)

    with np.printoptions(precision=3, suppress=True, linewidth=120):
        print("Target U:\n", U_target.imag)

    qc_impl = QuantumCircuit(4)
    # qargs in order [0,1,2,3]
    hw_phases = NQubitDecomposer.decompose_general_unitary(qc_impl, U_target, [0, 1, 2, 3])

    U_impl = Operator(qc_impl).data

    # 3) Print both matrices (rounded) for inspection
    with np.printoptions(precision=3, suppress=True, linewidth=120):
        print("=== Expected U (exp(i θ SWAP_01 SWAP_23)) ===")
        # print(U_target.imag)
        print_nontrivial_unitary_basis_action(U_target, n_qubits=4)
        print("\n=== Implemented U (from EC decomposer) ===")
        # print(U_impl.imag)
        print_nontrivial_unitary_basis_action(U_impl, n_qubits=4)
        print("\n=== Implemented U (real part) ===")
        # print(U_impl.real)
        # print("\n=== Implemented U (real part) ===")
        # print(U_impl.real)

    # 4) Check they match up to global phase
    assert _unitaries_close_up_to_phase(U_impl, U_target), \
        "exp(i θ SWAP_01 SWAP_23) decomposition failed!"


# ---------------------------------------------------------
# Run directly
# ---------------------------------------------------------
test_decompose_exp_i_theta_swap12_swap34()

Target U:
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]

Decomposing Hamming weight 1 block (size 4) [(4, 4)]: Phase in degrees: 0.000
2-level block on indices 0,1 has determinant 1.000+0.000j
2-level block on indices 2,3 has de

ValueError: cannot reshape array of size 1024 into shape (256,)

In [ ]:
def make_3q_two_level_unitary(n_qubits=4, i=1, j=2):
    """
    Construct an 8x8 unitary that is identity except for a 2x2 SU(2) block
    mixing |001> (index 1) and |010> (index 2).

    Returns:
        U: 8x8 np.ndarray complex
    """
    # Generic SU(2) block
    theta = 3 * np.pi / 2
    phi = 3 * np.pi / 2

    c = np.cos(theta)
    s = np.sin(theta)

    
    sub = np.array(
        [
            [0, 1j],
            [-1j, 0]
        ],
        dtype=complex
    )

    # Full 3-qubit unitary: identity except this 2x2 block
    U = np.eye(2**n_qubits, dtype=complex)

    # basis: |000>=0, |001>=1, |010>=2, |011>=3,
    #        |100>=4, |101>=5, |110>=6, |111>=7
    # i = 1  # |001>
    # j = 2  # |010>

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    # print the euler angles of the submatrix
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    with np.printoptions(precision=3, suppress=True):
        print("Submatrix:")
        print(sub)
        print(f"Euler angles (XYX) of submatrix: theta={theta_x}, phi={phi_y}, lambda={lam_x}, global_phase={global_phase}")
        print(f"U :")
        print(U)

    return U

def test_simple_2level_hd2():
    n_qubits = 7
    dim = 2**n_qubits
        
    U_target = make_3q_two_level_unitary(n_qubits=n_qubits, i=1, j=2)

    print(f"Target unitary (dim={U_target.shape}) (2-level, HD=2):")
    print_nontrivial_unitary_basis_action(U_target, n_qubits)
      
    # Decompose it
    qc = QuantumCircuit(n_qubits)
    NQubitDecomposer.decompose_general_unitary(qc, U_target, list(range(n_qubits)))
    
    # Check result
    U_impl = Operator(qc).data
    
    print("\nImplemented unitary:")
    print_nontrivial_unitary_basis_action(U_impl, n_qubits+1)
    # with np.printoptions(precision=3, suppress=True, linewidth=200):
    #     print("Real part:")
    #     print(U_impl.real)
    #     print("\nImaginary part:")
    #     print(U_impl.imag)
    
    U_impl = get_effective_unitary(qc, ancilla_indices=[n_qubits], ancilla_state=0)
    print(f"Shapes: U_target {U_target.shape}, U_impl {U_impl.shape}")
    match = _unitaries_close_up_to_phase(U_target, U_impl, atol=1e-6)
    print(f"\nMatch: {match}")
    
    if not match:
        with np.printoptions(precision=3, suppress=True, linewidth=200):
            print("\nDifference:")
            diff = U_target - U_impl
            print(f"Max diff: {np.max(np.abs(diff)):.6e}")
            print("Difference matrix (real):")
            with np.printoptions(precision=3, suppress=True):
                print(diff.real)
            print("Difference matrix (imag):")
            with np.printoptions(precision=3, suppress=True):
                print(diff.imag)
            
            print("\nImplemented unitary debug info:")
            print_nontrivial_unitary_basis_action(U_impl, n_qubits)
    
    assert match, "HD=2 case should work!"
    print("✓ Test passed")

test_simple_2level_hd2()

Submatrix:
[[ 0.+0.j  0.+1.j]
 [-0.-1.j  0.+0.j]]
Euler angles (XYX) of submatrix: theta=3.141592653589793, phi=1.5707963267948966, lambda=1.5707963267948966, global_phase=4.71238898038469
U :
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+1.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -0.-1.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j]]
Target unitary (dim=(8, 8)) (2-level, HD=2):
|001> -> -1.000j|010>
|010> -> +1.000j|001>

Decomposing Hamming weight 1 block (size 3) [(3, 3)]: Phase in degrees: 60.000
2-level block on indices 0,1 has determinant 0.500+0.866j
  Extracted SU(3) block:
[[ 0.   +0.j     0

AssertionError: HD=2 case should work!

In [99]:
def make_3q_two_level_unitary(n_qubits=4, i=1, j=2, theta=np.pi/2, phi=np.pi/4):
    # Generic SU(2) block
    theta = theta
    phi = phi

    c = np.cos(theta)
    s = np.sin(theta)

    sub = np.array(
        [
            [c, np.exp(1j * phi) * s],
            [-np.exp(-1j * phi) * s, c]
        ],
        dtype=complex
    )

    # Full 3-qubit unitary: identity except this 2x2 block
    U = np.eye(2**n_qubits, dtype=complex)

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    # print the euler angles of the submatrix
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    with np.printoptions(precision=3, suppress=True):
        print("Submatrix:")
        print(sub)
        print(f"Euler angles (XYX) of submatrix: theta={theta_x}, phi={phi_y}, lambda={lam_x}, global_phase={global_phase}")
    return U

def test_simple_2level_hd2(theta, phi):
    n_qubits = 3
    dim = 2**n_qubits
        
    U_target = make_3q_two_level_unitary(n_qubits=n_qubits, i=3, j=6, theta=theta, phi=phi)

    print(f"Target unitary (dim={U_target.shape}) (2-level, HD=2):")
    print_nontrivial_unitary_basis_action(U_target, n_qubits)
      
    # Decompose it
    qc = QuantumCircuit(n_qubits)
    NQubitDecomposer.decompose_general_unitary(qc, U_target, list(range(n_qubits)))
    
    # Check result
    U_impl = Operator(qc).data
    
    print("\nImplemented unitary:")
    print_nontrivial_unitary_basis_action(U_impl, n_qubits+1)
    # with np.printoptions(precision=3, suppress=True, linewidth=200):
    #     print("Real part:")
    #     print(U_impl.real)
    #     print("\nImaginary part:")
    #     print(U_impl.imag)
    
    U_impl = get_effective_unitary(qc, ancilla_indices=[n_qubits], ancilla_state=0)
    print(f"Shapes: U_target {U_target.shape}, U_impl {U_impl.shape}")
    match = _unitaries_close_up_to_phase(U_target, U_impl, atol=1e-6)
    print(f"\nMatch: {match}")
    
    if not match:
        with np.printoptions(precision=3, suppress=True, linewidth=200):
            print("\nDifference:")
            diff = U_target - U_impl
            print(f"Max diff: {np.max(np.abs(diff)):.6e}")
            # print("Difference matrix (real):")
            # with np.printoptions(precision=3, suppress=True):
            #     print(diff.real)
            # print("Difference matrix (imag):")
            # with np.printoptions(precision=3, suppress=True):
            #     print(diff.imag)
            
            print("\nImplemented unitary debug info:")
            print_nontrivial_unitary_basis_action(U_impl, n_qubits)
    
    assert match, "HD=2 case should work!"
    print("✓ Test passed")
    return match

test_simple_2level_hd2(theta=np.pi/3, phi=np.pi/4)

Submatrix:
[[ 0.5  +0.j     0.612+0.612j]
 [-0.612+0.612j  0.5  +0.j   ]]
Euler angles (XYX) of submatrix: theta=1.3181160716528182, phi=2.2555155297971794, lambda=2.2555155297971794, global_phase=3.141592653589793
Target unitary (dim=(8, 8)) (2-level, HD=2):
|011> -> +0.500|011> + (-0.612+0.612j)|110>
|110> -> (+0.612+0.612j)|011> + +0.500|110>

Decomposing Hamming weight 1 block (size 3) [(3, 3)]: Phase in degrees: 0.000
  Extracted SU(3) block:
[[1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j]]
  Extracted 0 two-level blocks:

Decomposing Hamming weight 2 block (size 3) [(3, 3)]: Phase in degrees: 0.000
2-level block on indices 0,2 has determinant 1.000+0.000j
  Extracted SU(3) block:
[[ 0.5  -0.j     0.   +0.j     0.612+0.612j]
 [ 0.   +0.j     1.   -0.j     0.   +0.j   ]
 [-0.612+0.612j  0.   +0.j     0.5  -0.j   ]]
  Extracted 1 two-level blocks:
  TLU between local indices 0, 2 with submatrix:
[[ 0.5  -0.j     0.612+0.612j]
 [-0.612+0.612j  0.5  -0.j   ]]
 

AssertionError: HD=2 case should work!